# Download de Mosaicos de Satélite a partir de CSV
Lê um arquivo CSV com colunas: `Carimbo de data/hora`, `Coordenadas da Bounding Box`, `Nome do Bairro`
e baixa imagens de satélite (ESRI World Imagery) para cada bairro.

In [ ]:
import math, os, io, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
import requests
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# ── CONFIGURAÇÕES ─────────────────────────────────────────────────────────────
CSV_PATH = "cordenadasheli.csv"   # <-- coloque aqui o caminho do seu arquivo CSV
ZOOM = 19                  # zoom do tile (19 = máximo prático para imagens urbanas)
TILE_URL = "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}"
HEADERS  = {"User-Agent": "Projeto-Imagens-Satelite/1.0"}

In [ ]:
# ── LER O CSV ─────────────────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)
print(f"Linhas carregadas: {len(df)}")
print(df.head())

In [ ]:
# ── PARSEAR COORDENADAS ────────────────────────────────────────────────────────
# Coluna esperada: "lon_min  lat_min  lon_max  lat_max" (separado por tabs ou espaços)
def parse_bbox(raw):
    """Converte string '−43.659 −22.961 −43.658 −22.960' em tupla de floats."""
    parts = str(raw).replace(',', ' ').split()
    return tuple(float(p) for p in parts)   # lon_min, lat_min, lon_max, lat_max

df["bbox"] = df["Coordenadas da Bounding Box"].apply(parse_bbox)

# Garante nomes únicos de pasta (bairros podem se repetir no CSV)
name_count = {}
slugs = []
for nome in df["Nome do Bairro"]:
    slug = nome.strip().replace(" ", "_").replace("/", "-")
    n = name_count.get(slug, 0)
    name_count[slug] = n + 1
    slugs.append(slug if n == 0 else f"{slug}_{n+1}")

df["slug"] = slugs
print(df[["Nome do Bairro", "slug", "bbox"]].to_string())

In [ ]:
# ── CRIAR PASTAS DE SAÍDA ─────────────────────────────────────────────────────
OUT_DIRS = {}
for slug in df["slug"]:
    folder = Path(f"mosaico_{slug}")
    folder.mkdir(exist_ok=True)
    OUT_DIRS[slug] = folder
    print("Pasta criada:", folder.resolve())

In [ ]:
# ── FUNÇÕES DE CONVERSÃO TILE ─────────────────────────────────────────────────
def deg2tile(lat, lon, z):
    n = 2.0 ** z
    x = int((lon + 180.0) / 360.0 * n)
    lat_rad = math.radians(lat)
    y = int((1.0 - math.asinh(math.tan(lat_rad)) / math.pi) / 2.0 * n)
    return x, y

def deg_to_meters(lat_ref, dlat, dlon):
    return dlat * 111_132.0, dlon * 111_320.0 * math.cos(math.radians(lat_ref))

# Resumo de cada bairro
for _, row in df.iterrows():
    lon_min, lat_min, lon_max, lat_max = row["bbox"]
    x_min, y_max = deg2tile(lat_min, lon_min, ZOOM)
    x_max, y_min = deg2tile(lat_max, lon_max, ZOOM)
    n_cols = x_max - x_min + 1
    n_rows = y_max - y_min + 1
    h_m, w_m = deg_to_meters((lat_min+lat_max)/2, lat_max-lat_min, lon_max-lon_min)
    print(f"{row['slug']}: {n_cols}x{n_rows} tiles | "
          f"{n_cols*256}x{n_rows*256} px | "
          f"{w_m:.0f}m x {h_m:.0f}m")

In [ ]:
# ── DOWNLOAD DOS TILES ────────────────────────────────────────────────────────
def baixar_tile(args):
    z, x, y, out_dir = args
    path = out_dir / f"tile_z{z}_x{x}_y{y}.jpg"
    if path.exists():
        return path, True, 0
    for tentativa in range(3):
        try:
            r = requests.get(TILE_URL.format(z=z, x=x, y=y),
                             headers=HEADERS, timeout=25)
            if r.status_code == 200 and len(r.content) > 2521:
                path.write_bytes(r.content)
                return path, True, tentativa
        except Exception:
            time.sleep(0.5)
    return path, False, 3

for _, row in df.iterrows():
    slug = row["slug"]
    lon_min, lat_min, lon_max, lat_max = row["bbox"]
    out_dir = OUT_DIRS[slug]

    x_min, y_max = deg2tile(lat_min, lon_min, ZOOM)
    x_max, y_min = deg2tile(lat_max, lon_max, ZOOM)

    jobs = [(ZOOM, x, y, out_dir)
            for x in range(x_min, x_max + 1)
            for y in range(y_min, y_max + 1)]

    print(f"\n[{slug}] Baixando {len(jobs)} tiles...")
    t0 = time.time()
    ok = falhas = 0
    with ThreadPoolExecutor(max_workers=4) as ex:
        for path, sucesso, _ in ex.map(baixar_tile, jobs):
            if sucesso: ok += 1
            else:       falhas += 1

    dt = time.time() - t0
    mb = sum(p.stat().st_size for p in out_dir.glob('*.jpg')) / 1024 / 1024
    print(f"  OK: {ok}/{len(jobs)} | Falhas: {falhas} | "
          f"Tempo: {dt:.1f}s | Disco: {mb:.1f} MB")

In [ ]:
# ── AMOSTRA DE TILES (grade 3x3) ──────────────────────────────────────────────
for _, row in df.iterrows():
    slug = row["slug"]
    lon_min, lat_min, lon_max, lat_max = row["bbox"]
    out_dir = OUT_DIRS[slug]

    x_min, y_max = deg2tile(lat_min, lon_min, ZOOM)
    x_max, y_min = deg2tile(lat_max, lon_max, ZOOM)

    xs = np.linspace(x_min, x_max, 3, dtype=int)
    ys = np.linspace(y_min, y_max, 3, dtype=int)

    fig, axes = plt.subplots(3, 3, figsize=(9, 9))
    for i, y in enumerate(ys):
        for j, x in enumerate(xs):
            p = out_dir / f"tile_z{ZOOM}_x{x}_y{y}.jpg"
            ax = axes[i, j]
            if p.exists():
                ax.imshow(Image.open(p))
                ax.set_title(f"x={x}, y={y}", fontsize=8)
            ax.set_xticks([]); ax.set_yticks([])

    plt.suptitle(f"Amostras — {row['Nome do Bairro']} @ zoom {ZOOM}",
                 fontweight="bold", y=1.0)
    plt.tight_layout()
    plt.savefig(f"amostra_tiles_{slug}.png", dpi=110, bbox_inches="tight")
    plt.show()

In [ ]:
# ── MONTAR MOSAICO COMPLETO ───────────────────────────────────────────────────
TILE_PX = 256

for _, row in df.iterrows():
    slug = row["slug"]
    lon_min, lat_min, lon_max, lat_max = row["bbox"]
    out_dir = OUT_DIRS[slug]

    x_min, y_max = deg2tile(lat_min, lon_min, ZOOM)
    x_max, y_min = deg2tile(lat_max, lon_max, ZOOM)
    n_cols = x_max - x_min + 1
    n_rows = y_max - y_min + 1
    W, H = n_cols * TILE_PX, n_rows * TILE_PX

    mosaico = Image.new("RGB", (W, H), (0, 0, 0))
    faltando = 0
    for x in range(x_min, x_max + 1):
        for y in range(y_min, y_max + 1):
            p = out_dir / f"tile_z{ZOOM}_x{x}_y{y}.jpg"
            if p.exists():
                mosaico.paste(Image.open(p),
                              ((x - x_min) * TILE_PX, (y - y_min) * TILE_PX))
            else:
                faltando += 1

    print(f"{slug}: {W}x{H} px | {faltando} tile(s) faltando")

    mosaico.save(f"mosaico_{slug}_hires_full.jpg", quality=88)

    preview = mosaico.copy()
    preview.thumbnail((3000, 3000))
    preview.save(f"mosaico_{slug}_hires_preview.jpg", quality=85)

    plt.figure(figsize=(12, 12 * H / W))
    plt.imshow(preview)
    plt.title(
        f"Mosaico de {row['Nome do Bairro']} @ z={ZOOM} — {n_cols}x{n_rows} tiles ({W}x{H} px)\n"
        "Fonte: Esri, Maxar, Earthstar Geographics, and the GIS User Community",
        fontsize=10
    )
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(f"mosaico_{slug}_hires_anotado.png", dpi=110, bbox_inches="tight")
    plt.show()